# 01 — Data Exploration: PubMedQA for Hormonal Health QA

This notebook loads the PubMedQA dataset, filters it to the women's/hormonal health domain, and explores the data characteristics.

**Dataset**: [qiaojin/PubMedQA](https://huggingface.co/datasets/qiaojin/PubMedQA)  
**Domain Focus**: Hormonal health, reproductive health, women's health conditions (PCOS, endometriosis, menstrual health, etc.)

In [ ]:
# Install dependencies (run in Colab)
# !pip install -q datasets transformers pandas matplotlib seaborn

In [ ]:
import sys
sys.path.append('..')

from src.data_utils import (
    load_pubmedqa,
    filter_health_domain,
    dataset_to_dataframe,
    get_split_stats,
    HEALTH_KEYWORDS,
)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.max_colwidth', 120)

## 1. Load the Expert-Annotated Subset (pqa_labeled)

In [ ]:
# Load expert-labeled subset (~1K examples with gold yes/no/maybe labels)
labeled_ds = load_pubmedqa('pqa_labeled')
print(f"Labeled split: {labeled_ds}")
print(f"Number of examples: {len(labeled_ds['train'])}")

In [ ]:
# Inspect one example
example = labeled_ds['train'][0]
print(f"Keys: {list(example.keys())}")
print(f"\nQuestion: {example['question']}")
print(f"\nFinal Decision: {example['final_decision']}")
print(f"\nLong Answer: {example['long_answer'][:300]}...")
print(f"\nContext keys: {list(example['context'].keys()) if isinstance(example['context'], dict) else type(example['context'])}")

## 2. Filter to Hormonal/Women's Health Domain

In [ ]:
# Filter labeled subset
health_labeled = filter_health_domain(labeled_ds['train'])
print(f"Health-domain examples (labeled): {len(health_labeled)} / {len(labeled_ds['train'])}")
print(f"Percentage: {len(health_labeled)/len(labeled_ds['train'])*100:.1f}%")

In [ ]:
# Also load artificial subset for building a larger retrieval corpus
artificial_ds = load_pubmedqa('pqa_artificial')
print(f"Artificial split: {artificial_ds}")
print(f"Number of examples: {len(artificial_ds['train'])}")

health_artificial = filter_health_domain(artificial_ds['train'])
print(f"\nHealth-domain examples (artificial): {len(health_artificial)} / {len(artificial_ds['train'])}")

## 3. Explore the Filtered Dataset

In [ ]:
# Convert to DataFrame for analysis
df_labeled = dataset_to_dataframe(health_labeled)
print(f"Shape: {df_labeled.shape}")
df_labeled.head()

In [ ]:
# Label distribution
stats = get_split_stats(df_labeled)
print("Dataset Statistics:")
for key, val in stats.items():
    print(f"  {key}: {val}")

In [ ]:
# Visualize label distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Label counts
df_labeled['final_decision'].value_counts().plot(
    kind='bar', ax=axes[0], color=['#4CAF50', '#F44336', '#FF9800']
)
axes[0].set_title('Answer Distribution (Health Domain)')
axes[0].set_xlabel('Decision')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Text length distributions
df_labeled['question_len'] = df_labeled['question'].str.len()
df_labeled['context_len'] = df_labeled['context'].str.len()
df_labeled[['question_len', 'context_len']].plot(
    kind='hist', bins=30, alpha=0.7, ax=axes[1]
)
axes[1].set_title('Text Length Distribution')
axes[1].set_xlabel('Character Length')

plt.tight_layout()
plt.savefig('../figures/data_exploration.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Show sample questions
print("Sample health-domain questions:\n")
for i, row in df_labeled.head(10).iterrows():
    print(f"  [{row['final_decision']:>5}] {row['question']}")

In [ ]:
# Keyword frequency analysis — which health topics are most represented?
from collections import Counter

keyword_counts = Counter()
all_text = (df_labeled['question'] + ' ' + df_labeled['context']).str.lower()

keyword_groups = {
    'PCOS/Ovarian': ['pcos', 'polycystic', 'ovarian', 'ovary'],
    'Endometriosis': ['endometriosis', 'endometrial'],
    'Menstrual': ['menstrual', 'menstruation', 'dysmenorrhea', 'amenorrhea'],
    'Hormones': ['estrogen', 'progesterone', 'testosterone', 'hormonal', 'hormone'],
    'Fertility': ['fertility', 'infertility', 'ovulation'],
    'Pregnancy': ['pregnancy', 'pregnant', 'prenatal'],
    'Menopause': ['menopause', 'perimenopause', 'postmenopause'],
    'Contraception': ['contracepti', 'oral contraceptive'],
    'Breast': ['breast cancer', 'breastfeeding'],
    'Thyroid': ['thyroid'],
}

topic_counts = {}
for topic, keywords in keyword_groups.items():
    count = sum(all_text.str.contains('|'.join(keywords), regex=True))
    topic_counts[topic] = count

topic_df = pd.Series(topic_counts).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
topic_df.plot(kind='barh', ax=ax, color='#7B1FA2')
ax.set_title('Health Topic Distribution in Filtered Dataset')
ax.set_xlabel('Number of Examples')
plt.tight_layout()
plt.savefig('../figures/topic_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Prepare Data Splits

We'll use the expert-labeled health examples for evaluation (test set) and the artificial health examples for building the retrieval corpus.

In [ ]:
# Save filtered data for next notebooks
import json

# Save health-domain examples
with open('../data/sample/health_labeled.json', 'w') as f:
    json.dump(health_labeled[:50], f, indent=2)  # Small sample for submission

print(f"Saved {min(50, len(health_labeled))} labeled examples to data/sample/")
print(f"\nSummary:")
print(f"  Labeled (eval): {len(health_labeled)} examples")
print(f"  Artificial (retrieval corpus): {len(health_artificial)} examples")
print(f"  Total health-domain: {len(health_labeled) + len(health_artificial)} examples")

## Next Steps

1. **Notebook 02**: Build the retrieval pipeline (BiomedBERT embeddings + FAISS index)
2. **Notebook 03**: Full RAG pipeline (retrieval + BioMistral-7B generation)
3. **Notebook 04**: Evaluation and visualizations